# RepOD Pretrained Models on Colab (`0.2.1`)

This notebook downloads pretrained models from RepOD and validates them with release `0.2.1` of this repository.

It includes two concrete examples:
- `linear-bottom-2024.zip`
- `transformer-bottom-base-2024.zip`

Sources:
- Dataset DOI: [10.18150/OCUTSI](https://doi.org/10.18150/OCUTSI)
- RepOD dataset page: [Job offers classifiers for ISCO and KZiS 2023](https://repod.icm.edu.pl/dataset.xhtml?persistentId=doi:10.18150/OCUTSI)
- Dataverse Native API docs: [dataset metadata endpoint](https://guides.dataverse.org/en/5.13/api/native-api.html)
- Dataverse Data Access API docs: [file download endpoint](https://guides.dataverse.org/en/5.3/api/dataaccess.html)


In [ ]:
%cd /content
!rm -rf /content/job-ads-classifier /content/repod-models
!git clone --branch v0.2.1 https://github.com/OJALAB/job-ads-classifier.git /content/job-ads-classifier
%cd /content/job-ads-classifier
!python -m pip install --upgrade pip
!pip install -r requirements-colab-0.2.0.txt


In [ ]:
!python main.py --help
!pytest -m "not integration"


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import urllib.parse
import urllib.request
import zipfile
from pathlib import Path

SERVER = "https://repod.icm.edu.pl"
DATASET_PID = "doi:10.18150/OCUTSI"
DOWNLOAD_DIR = Path("/content/repod-models/downloads")
EXTRACT_DIR = Path("/content/repod-models/extracted")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

def list_dataset_files(server=SERVER, dataset_pid=DATASET_PID):
    query = urllib.parse.urlencode({"persistentId": dataset_pid})
    url = f"{server}/api/datasets/:persistentId/versions/:latest-published?{query}"
    with urllib.request.urlopen(url) as response:
        payload = json.load(response)
    return payload["data"]["files"]

def find_file_metadata(filename, server=SERVER, dataset_pid=DATASET_PID):
    for item in list_dataset_files(server=server, dataset_pid=dataset_pid):
        datafile = item["dataFile"]
        if datafile["filename"] == filename:
            return {
                "id": datafile["id"],
                "filename": datafile["filename"],
                "filesize": datafile.get("filesize"),
                "md5": datafile.get("md5"),
                "contentType": datafile.get("contentType"),
            }
    raise FileNotFoundError(f"Could not find file: {filename}")

def md5sum(path):
    digest = hashlib.md5()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def download_repod_file(filename, out_dir=DOWNLOAD_DIR, server=SERVER, dataset_pid=DATASET_PID):
    meta = find_file_metadata(filename, server=server, dataset_pid=dataset_pid)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / filename
    url = f"{server}/api/access/datafile/{meta['id']}"

    with urllib.request.urlopen(url) as response, open(out_path, "wb") as handle:
        shutil.copyfileobj(response, handle)

    local_md5 = md5sum(out_path)
    print({**meta, "local_md5": local_md5, "path": str(out_path)})
    if meta["md5"]:
        assert local_md5 == meta["md5"], f"MD5 mismatch for {filename}: {local_md5} != {meta['md5']}"
    return out_path, meta

def extract_zip(zip_path, target_dir):
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(target_dir)
    return target_dir

def find_model_dir(root_dir):
    root_dir = Path(root_dir)
    candidates = sorted({path.parent for path in root_dir.rglob("hierarchy.bin")})
    assert candidates, f"No extracted model directory found in {root_dir}"
    return str(candidates[0])

def run_command(command, env=None):
    print("$", " ".join(command))
    result = subprocess.run(command, env=env, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result

print("Available files preview:")
for item in list_dataset_files()[:6]:
    print(item["dataFile"]["filename"])


## Example 1: `linear-bottom-2024.zip`

RepOD lists this file as:
- filename: `linear-bottom-2024.zip`
- size: 136.2 MB
- MD5: `dc101c4d2411446c2d63e49bef78b66c`

Source: [RepOD dataset page](https://repod.icm.edu.pl/dataset.xhtml?persistentId=doi:10.18150/OCUTSI)


In [ ]:
linear_zip, linear_meta = download_repod_file("linear-bottom-2024.zip")
linear_root = extract_zip(linear_zip, EXTRACT_DIR / "linear-bottom-2024")
linear_model_dir = find_model_dir(linear_root)
print("LINEAR_MODEL_DIR =", linear_model_dir)

In [ ]:
linear_env = os.environ.copy()
linear_env["EXISTING_MODEL_CLASSIFIER"] = "LinearJobOffersClassifier"
linear_env["EXISTING_MODEL_DIR"] = linear_model_dir
linear_env["EXISTING_MODEL_INPUT"] = "/content/job-ads-classifier/tests/data/x_test.txt"

run_command(["pytest", "tests/test_existing_model_cpu.py", "-q"], env=linear_env)

In [ ]:
run_command([
    "python",
    "main.py",
    "predict",
    "LinearJobOffersClassifier",
    "-x",
    "tests/data/x_test.txt",
    "-m",
    linear_model_dir,
    "-p",
    "/content/predictions-linear.txt",
    "-T",
    "1",
])

run_command(["head", "-n", "5", "/content/predictions-linear.txt"])
run_command(["head", "-n", "5", "/content/predictions-linear.txt.map"])


## Example 2: `transformer-bottom-base-2024.zip`

RepOD lists this file as:
- filename: `transformer-bottom-base-2024.zip`
- size: 1.6 GB
- MD5: `1677a4e5395effae7dc7f005fd7f35ed`

Source: [RepOD dataset page](https://repod.icm.edu.pl/dataset.xhtml?persistentId=doi:10.18150/OCUTSI)

This section is heavier than the linear example because the archive is much larger.


In [ ]:
transformer_zip, transformer_meta = download_repod_file("transformer-bottom-base-2024.zip")
transformer_root = extract_zip(transformer_zip, EXTRACT_DIR / "transformer-bottom-base-2024")
transformer_model_dir = find_model_dir(transformer_root)
print("TRANSFORMER_MODEL_DIR =", transformer_model_dir)

In [ ]:
transformer_env = os.environ.copy()
transformer_env["EXISTING_MODEL_CLASSIFIER"] = "TransformerJobOffersClassifier"
transformer_env["EXISTING_MODEL_DIR"] = transformer_model_dir
transformer_env["EXISTING_MODEL_INPUT"] = "/content/job-ads-classifier/tests/data/x_test.txt"

run_command(["pytest", "tests/test_existing_model_cpu.py", "-q"], env=transformer_env)

In [ ]:
run_command([
    "python",
    "main.py",
    "predict",
    "TransformerJobOffersClassifier",
    "-x",
    "tests/data/x_test.txt",
    "-m",
    transformer_model_dir,
    "-p",
    "/content/predictions-transformer.txt",
    "-A",
    "cpu",
    "-P",
    "32",
    "-T",
    "1",
])

run_command(["head", "-n", "5", "/content/predictions-transformer.txt"])
run_command(["head", "-n", "5", "/content/predictions-transformer.txt.map"])
